In [2]:
from ultralytics import YOLO
model = YOLO("yolo26n.pt")


In [ ]:
import cv2
import numpy as np

# 1. Load the network
net = cv2.dnn.readNet("yolov3.weights", "yolov3.cfg")
classes = open("coco.names").read().strip().split("\n")
layer_names = net.getLayerNames()
output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers()]

# 2. Load and preprocess image
img = cv2.imread("image.jpg")
height, width, _ = img.shape

# Create a blob: (image, scale_factor, size, mean_subtraction, swapRB, crop)
blob = cv2.dnn.blobFromImage(img, 1/255.0, (416, 416), (0,0,0), swapRB=True, crop=False)
net.setInput(blob)

# 3. Run Inference
outs = net.forward(output_layers)

# 4. Process Outputs
for out in outs:
    for detection in out:
        scores = detection[5:]
        class_id = np.argmax(scores)
        confidence = scores[class_id]
        if confidence > 0.5:
            # Object detected - calculate coordinates
            center_x = int(detection[0] * width)
            center_y = int(detection[1] * height)
            w = int(detection[2] * width)
            h = int(detection[3] * height)
            
            # Rectangle coordinates
            x = int(center_x - w / 2)
            y = int(center_y - h / 2)
            cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(img, classes[class_id], (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

cv2.imshow("Image", img)
cv2.waitKey(0)

In [ ]:
import time
import serial

model = YOLO(r'yolo26n.pt') 
cap = cv2.VideoCapture(0)

# ตัวแปรสำหรับคำนวณ FPS
prev_time = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 1. คำนวณ FPS
    current_time = time.time()
    fps = 1 / (current_time - prev_time)
    prev_time = current_time

    # 2. Predict เฉพาะคน (classes=[0])
    results = model(frame, stream=True, classes=[0], max_det=8)
    print("Number of class :", len(results[0].boxes))

    for r in results:
        # นับจำนวนคน
        person_count = len(r.boxes)
        
        # วาด Box
        annotated_frame = r.plot()
        
        # 3. แสดง FPS บนหน้าจอ (มุมซ้ายบน)
        cv2.putText(annotated_frame, f'FPS: {int(fps)}', (20, 40), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
        
        # 4. แสดงจำนวนคน
        cv2.putText(annotated_frame, f'People: {person_count}', (20, 80), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        cv2.imshow("YOLOv8 - Person Counter & FPS", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



SERIAL_PORT_NAME = "COM7"  # Change this to your serial port name
BAUD_RATE = 9600

try:
    ser = serial.Serial(SERIAL_PORT_NAME, BAUD_RATE, timeout=1)
    time.sleep(2)  # Wait for the connection to initialize
    print(f"Connected to {SERIAL_PORT_NAME} at {BAUD_RATE} baud.")

    data = len(results[0].boxes)

    ser.write(data.encode('utf-8'))  # Send data to the serial port
    print(f"Sent data: '{data.strip()}'")

    while True:
        if ser.in_waiting > 0:
            response = ser.readline().decode('utf-8').strip()
            print(f"Received from serial: '{response}'")

except serial.SerialException as e:
    print(f"Serial error: {e}")

except KeyboardInterrupt:
    print("Exiting program.")
finally:
    if ser and ser.is_open:
        ser.close()
        print(f"Closed serial connection to {SERIAL_PORT_NAME}.")            